# Run 322 artifact probe

Draft notebook for `run_id=322b4485-557c-434b-b2e1-0776d15a515b`.

This notebook loads the pinned BTCUSDT `1h` price artifacts and the two signal matrices used by the run:

- `ma.dema` from `signals/1h/ma.dema/signals.i8.npy`
- `ma.ema` from `signals/1h/ma.ema/signals.i8.npy`

Pinned runtime identity:

- slot: `slot_a`
- generation: `1`
- asof_date: `2026-04-02`
- manifest_hash: `13df35a144a9706b6b40c949f71ddc5dd23d60da10bbda81c12e26f9676faaae`
- timeframe: `1h`
- symbol: `BTCUSDT`


In [1]:
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import yaml

RUN_ID = "322b4485-557c-434b-b2e1-0776d15a515b"
TIMEFRAME = "1h"
RUN_TIME_RANGE_START = datetime(2017, 10, 6, 22, 55, tzinfo=timezone.utc)
RUN_TIME_RANGE_END = datetime(2026, 3, 31, 22, 55, tzinfo=timezone.utc)
RUN_TIME_RANGE_START_MS = int(RUN_TIME_RANGE_START.timestamp() * 1000)
RUN_TIME_RANGE_END_MS = int(RUN_TIME_RANGE_END.timestamp() * 1000)

ARTIFACT_ROOT = Path("/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a")
PRICE_DIR = ARTIFACT_ROOT / "prices" / TIMEFRAME
DEMA_DIR = ARTIFACT_ROOT / "signals" / TIMEFRAME / "ma.dema"
EMA_DIR = ARTIFACT_ROOT / "signals" / TIMEFRAME / "ma.ema"

SLOT_MANIFEST_PATH = ARTIFACT_ROOT / "manifest.yaml"
PRICE_OPEN_TIME_PATH = PRICE_DIR / "open_time.i64.npy"
PRICE_CLOSE_TIME_PATH = PRICE_DIR / "close_time.i64.npy"
PRICE_OHLCV_PATH = PRICE_DIR / "ohlcv.f32.npy"
DEMA_MANIFEST_PATH = DEMA_DIR / "manifest.yaml"
EMA_MANIFEST_PATH = EMA_DIR / "manifest.yaml"
DEMA_SIGNAL_PATH = DEMA_DIR / "signals.i8.npy"
EMA_SIGNAL_PATH = EMA_DIR / "signals.i8.npy"

REQUEST_INDICATOR_GRIDS = [
    {
        "indicator_id": "ma.dema",
        "sources": ["close"],
        "window_range": [5, 200],
    },
    {
        "indicator_id": "ma.ema",
        "sources": ["high", "ohlc4"],
        "window_range": [5, 200],
    },
]

for path in [
    SLOT_MANIFEST_PATH,
    PRICE_OPEN_TIME_PATH,
    PRICE_CLOSE_TIME_PATH,
    PRICE_OHLCV_PATH,
    DEMA_MANIFEST_PATH,
    EMA_MANIFEST_PATH,
    DEMA_SIGNAL_PATH,
    EMA_SIGNAL_PATH,
]:
    print(f"{path}: exists={path.exists()}")


/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a/manifest.yaml: exists=True
/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a/prices/1h/open_time.i64.npy: exists=True
/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a/prices/1h/close_time.i64.npy: exists=True
/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a/prices/1h/ohlcv.f32.npy: exists=True
/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a/signals/1h/ma.dema/manifest.yaml: exists=True
/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a/signals/1h/ma.ema/manifest.yaml: exists=True
/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a/signals/1h/ma.dema/signals.i8.npy: exists=True
/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a/signals/1h/ma.ema/signals.i8.npy: exists=True


In [2]:
slot_manifest = yaml.safe_load(SLOT_MANIFEST_PATH.read_text())
dema_manifest = yaml.safe_load(DEMA_MANIFEST_PATH.read_text())
ema_manifest = yaml.safe_load(EMA_MANIFEST_PATH.read_text())

prices_by_timeframe = {item["timeframe"]: item for item in slot_manifest["prices"]}
price_manifest = prices_by_timeframe[TIMEFRAME]

print("slot:", slot_manifest["slot"], "generation:", slot_manifest["slot_generation"], "asof_date:", slot_manifest["asof_date"])
print("price bars:", price_manifest["coverage"]["bar_count"])
print("dema signals shape:", tuple(dema_manifest["signals"]["shape"]))
print("ema signals shape:", tuple(ema_manifest["signals"]["shape"]))
print("request indicator grids:", REQUEST_INDICATOR_GRIDS)


slot: slot_a generation: 1 asof_date: 2026-04-02
price bars: 75453
dema signals shape: (1176, 75453)
ema signals shape: (1176, 75453)
request indicator grids: [{'indicator_id': 'ma.dema', 'sources': ['close'], 'window_range': [5, 200]}, {'indicator_id': 'ma.ema', 'sources': ['high', 'ohlc4'], 'window_range': [5, 200]}]


In [3]:
price_open_time = np.load(PRICE_OPEN_TIME_PATH, mmap_mode="r")
price_close_time = np.load(PRICE_CLOSE_TIME_PATH, mmap_mode="r")
price_ohlcv = np.load(PRICE_OHLCV_PATH, mmap_mode="r")
dema_signals = np.load(DEMA_SIGNAL_PATH, mmap_mode="r")
ema_signals = np.load(EMA_SIGNAL_PATH, mmap_mode="r")

print("price_open_time:", price_open_time.shape, price_open_time.dtype)
print("price_close_time:", price_close_time.shape, price_close_time.dtype)
print("price_ohlcv:", price_ohlcv.shape, price_ohlcv.dtype)
print("dema_signals:", dema_signals.shape, dema_signals.dtype)
print("ema_signals:", ema_signals.shape, ema_signals.dtype)


price_open_time: (75453,) int64
price_close_time: (75453,) int64
price_ohlcv: (75453, 5) float32
dema_signals: (1176, 75453) int8
ema_signals: (1176, 75453) int8


In [4]:
open_time_index = price_open_time.astype("datetime64[ms]")
close_time_index = price_close_time.astype("datetime64[ms]")
time_mask = (price_open_time >= RUN_TIME_RANGE_START_MS) & (price_open_time <= RUN_TIME_RANGE_END_MS)

print("bars inside run time range:", int(time_mask.sum()))
print("first matching bar:", np.datetime_as_string(open_time_index[time_mask][0], timezone="UTC"))
print("last matching bar:", np.datetime_as_string(open_time_index[time_mask][-1], timezone="UTC"))


bars inside run time range: 74200
first matching bar: 2017-10-06T23:00:00.000Z
last matching bar: 2026-03-31T22:00:00.000Z


In [5]:
sample_indices = np.flatnonzero(time_mask)[:5]
sample_prices = [
    {
        "open_time": np.datetime_as_string(open_time_index[idx], timezone="UTC"),
        "close_time": np.datetime_as_string(close_time_index[idx], timezone="UTC"),
        "open": float(price_ohlcv[idx, 0]),
        "high": float(price_ohlcv[idx, 1]),
        "low": float(price_ohlcv[idx, 2]),
        "close": float(price_ohlcv[idx, 3]),
        "volume": float(price_ohlcv[idx, 4]),
    }
    for idx in sample_indices
]
sample_prices


[{'open_time': np.str_('2017-10-06T23:00:00.000Z'),
  'close_time': np.str_('2017-10-07T00:00:00.000Z'),
  'open': 4374.60009765625,
  'high': 4399.0,
  'low': 4343.330078125,
  'close': 4369.0,
  'volume': 9.739876747131348},
 {'open_time': np.str_('2017-10-07T00:00:00.000Z'),
  'close_time': np.str_('2017-10-07T01:00:00.000Z'),
  'open': 4369.0,
  'high': 4399.0,
  'low': 4369.0,
  'close': 4391.97021484375,
  'volume': 9.719825744628906},
 {'open_time': np.str_('2017-10-07T01:00:00.000Z'),
  'close_time': np.str_('2017-10-07T02:00:00.000Z'),
  'open': 4391.97021484375,
  'high': 4391.97021484375,
  'low': 4370.0,
  'close': 4380.5,
  'volume': 5.740303039550781},
 {'open_time': np.str_('2017-10-07T02:00:00.000Z'),
  'close_time': np.str_('2017-10-07T03:00:00.000Z'),
  'open': 4380.5,
  'high': 4389.0,
  'low': 4362.02001953125,
  'close': 4383.0,
  'volume': 14.227631568908691},
 {'open_time': np.str_('2017-10-07T03:00:00.000Z'),
  'close_time': np.str_('2017-10-07T04:00:00.000Z'),


In [6]:
signal_probe = [
    {
        "open_time": np.datetime_as_string(open_time_index[idx], timezone="UTC"),
        "dema_row_0": int(dema_signals[0, idx]),
        "ema_row_0": int(ema_signals[0, idx]),
    }
    for idx in range(12)
]
signal_probe


[{'open_time': np.str_('2017-08-17T07:00:00.000Z'),
  'dema_row_0': 0,
  'ema_row_0': 0},
 {'open_time': np.str_('2017-08-17T08:00:00.000Z'),
  'dema_row_0': 1,
  'ema_row_0': 1},
 {'open_time': np.str_('2017-08-17T09:00:00.000Z'),
  'dema_row_0': 1,
  'ema_row_0': 1},
 {'open_time': np.str_('2017-08-17T10:00:00.000Z'),
  'dema_row_0': 1,
  'ema_row_0': 1},
 {'open_time': np.str_('2017-08-17T11:00:00.000Z'),
  'dema_row_0': -1,
  'ema_row_0': 1},
 {'open_time': np.str_('2017-08-17T12:00:00.000Z'),
  'dema_row_0': -1,
  'ema_row_0': -1},
 {'open_time': np.str_('2017-08-17T13:00:00.000Z'),
  'dema_row_0': 1,
  'ema_row_0': 1},
 {'open_time': np.str_('2017-08-17T14:00:00.000Z'),
  'dema_row_0': 1,
  'ema_row_0': 1},
 {'open_time': np.str_('2017-08-17T15:00:00.000Z'),
  'dema_row_0': -1,
  'ema_row_0': -1},
 {'open_time': np.str_('2017-08-17T16:00:00.000Z'),
  'dema_row_0': -1,
  'ema_row_0': -1},
 {'open_time': np.str_('2017-08-17T17:00:00.000Z'),
  'dema_row_0': -1,
  'ema_row_0': -1},
 

## Notes

- The two `signals.i8.npy` files above are the exact artifact-backed matrices used by the run.
- Each file stores the full indicator matrix for one indicator family on the pinned `1h` timeline.
- The run-specific subset is narrower than the full matrix:
  - `ma.dema`: `source=close`, `window=5..200`
  - `ma.ema`: `source in {high, ohlc4}`, `window=5..200`
- This draft notebook intentionally stops at loading the exact pinned `npy` artifacts and probing timeline alignment.
- Next extension: reconstruct the row mapping for the selected sources/windows from the v2 signal rules/defaults catalog and then slice the relevant rows from each matrix.


## Additional 15m artifact loads for BTCUSDT spot

This section adds a broader `15m` probe on the same pinned slot for three extra indicators from different families:

- `ma.sma`
- `momentum.roc`
- `volatility.stddev`

It also loads:

- `prices/15m/*`
- derived price calculations (`open`, `high`, `low`, `close`, `hlc3`, `ohlc4`)
- strict `hit_times/15m/*` TP/SL tables and levels

The existing `1h` run-specific experiment above is kept intact.


In [7]:
TIMEFRAME_15M = "15m"
PRICE_DIR_15M = ARTIFACT_ROOT / "prices" / TIMEFRAME_15M
HIT_TIMES_DIR_1M = ARTIFACT_ROOT / "hit_times" / "1m"

EXTRA_INDICATORS_15M = {
    "ma.sma": {
        "family": "ma",
        "sources": ["close", "hlc3", "ohlc4", "low", "high", "open"],
        "param_name": "window",
        "param_values": list(range(5, 201)),
    },
    "momentum.roc": {
        "family": "momentum",
        "sources": ["close", "hlc3", "ohlc4", "low", "high", "open"],
        "param_name": "window",
        "param_values": [5, 7, 10, 14, 21, 28, 42, 63, 84, 126],
    },
    "volatility.stddev": {
        "family": "volatility",
        "sources": ["close", "hlc3", "ohlc4", "low", "high", "open"],
        "param_name": "window",
        "param_values": [10, 14, 20, 28, 42, 56, 84, 126],
        "signal_defaults": {"long_delta_periods": -5, "short_delta_periods": -10},
    },
}

PRICE_OPEN_TIME_15M_PATH = PRICE_DIR_15M / "open_time.i64.npy"
PRICE_CLOSE_TIME_15M_PATH = PRICE_DIR_15M / "close_time.i64.npy"
PRICE_OHLCV_15M_PATH = PRICE_DIR_15M / "ohlcv.f32.npy"
HIT_TIMES_MANIFEST_PATH = HIT_TIMES_DIR_1M / "manifest.yaml"
TP_VALUES_PATH = HIT_TIMES_DIR_1M / "tp_values.f32.npy"
SL_VALUES_PATH = HIT_TIMES_DIR_1M / "sl_values.f32.npy"
LONG_TP_PATH = HIT_TIMES_DIR_1M / "long_tp.u32.npy"
SHORT_TP_PATH = HIT_TIMES_DIR_1M / "short_tp.u32.npy"
LONG_SL_PATH = HIT_TIMES_DIR_1M / "long_sl.u32.npy"
SHORT_SL_PATH = HIT_TIMES_DIR_1M / "short_sl.u32.npy"

EXTRA_SIGNAL_PATHS_15M = {
    indicator_id: {
        "manifest": ARTIFACT_ROOT / "signals" / TIMEFRAME_15M / indicator_id / "manifest.yaml",
        "signals": ARTIFACT_ROOT / "signals" / TIMEFRAME_15M / indicator_id / "signals.i8.npy",
    }
    for indicator_id in EXTRA_INDICATORS_15M
}

paths_to_check = [
    PRICE_OPEN_TIME_15M_PATH,
    PRICE_CLOSE_TIME_15M_PATH,
    PRICE_OHLCV_15M_PATH,
    HIT_TIMES_MANIFEST_PATH,
    TP_VALUES_PATH,
    SL_VALUES_PATH,
    LONG_TP_PATH,
    SHORT_TP_PATH,
    LONG_SL_PATH,
    SHORT_SL_PATH,
] + [value for item in EXTRA_SIGNAL_PATHS_15M.values() for value in item.values()]

for path in paths_to_check:
    print(f"{path}: exists={path.exists()}")


/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a/prices/15m/open_time.i64.npy: exists=True
/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a/prices/15m/close_time.i64.npy: exists=True
/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a/prices/15m/ohlcv.f32.npy: exists=True
/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a/hit_times/15m/manifest.yaml: exists=True
/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a/hit_times/15m/tp_values.f32.npy: exists=True
/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a/hit_times/15m/sl_values.f32.npy: exists=True
/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a/hit_times/15m/long_tp.u32.npy: exists=True
/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a/hit_times/15m/short_tp.u32.npy: exists=True
/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a/hit_times/15m/long_sl.u32.npy: exists=True
/opt

In [8]:
price_manifest_15m = prices_by_timeframe[TIMEFRAME_15M]
signal_manifests_15m = {
    indicator_id: yaml.safe_load(paths["manifest"].read_text())
    for indicator_id, paths in EXTRA_SIGNAL_PATHS_15M.items()
}
hit_times_manifest = yaml.safe_load(HIT_TIMES_MANIFEST_PATH.read_text())

print("15m price bars:", price_manifest_15m["coverage"]["bar_count"])
for indicator_id, doc in signal_manifests_15m.items():
    print(
        indicator_id,
        "rows_count=", doc["rows_count"],
        "shape=", tuple(doc["signals"]["shape"]),
        "signal_defaults=", doc["grid"].get("signals_v1_params_defaults", {}),
    )
print("hit_times tables:", hit_times_manifest["tables"])
print("tp grid path:", hit_times_manifest["tp_values"])
print("sl grid path:", hit_times_manifest["sl_values"])


15m price bars: 301866
ma.sma rows_count= 1176 shape= (1176, 301866) signal_defaults= {}
momentum.roc rows_count= 60 shape= (60, 301866) signal_defaults= {}
volatility.stddev rows_count= 48 shape= (48, 301866) signal_defaults= {'long_delta_periods': -5, 'short_delta_periods': -10}
hit_times tables: {'long_tp': {'path': 'hit_times/15m/long_tp.u32.npy', 'dtype': 'uint32', 'shape': [5, 4528144], 'axis_order': ['level', 'time'], 'sha256': '1c6f9b3ee2da2ceaf9c280f5c5ea43cb38b3a866540d5230cc3643d11947b2d6', 'monotonicity': 'non_decreasing_by_level'}, 'long_sl': {'path': 'hit_times/15m/long_sl.u32.npy', 'dtype': 'uint32', 'shape': [5, 4528144], 'axis_order': ['level', 'time'], 'sha256': 'cd5752ee9f385e5837b2d887aefa62794fc8e99a64a1c6571dc6e42455c5edaf', 'monotonicity': 'non_decreasing_by_level'}, 'short_tp': {'path': 'hit_times/15m/short_tp.u32.npy', 'dtype': 'uint32', 'shape': [5, 4528144], 'axis_order': ['level', 'time'], 'sha256': 'cd5752ee9f385e5837b2d887aefa62794fc8e99a64a1c6571dc6e42455

In [9]:
price_open_time_15m = np.load(PRICE_OPEN_TIME_15M_PATH, mmap_mode="r")
price_close_time_15m = np.load(PRICE_CLOSE_TIME_15M_PATH, mmap_mode="r")
price_ohlcv_15m = np.load(PRICE_OHLCV_15M_PATH, mmap_mode="r")

signal_matrices_15m = {
    indicator_id: np.load(paths["signals"], mmap_mode="r")
    for indicator_id, paths in EXTRA_SIGNAL_PATHS_15M.items()
}

tp_values_1m = np.load(TP_VALUES_PATH, mmap_mode="r")
sl_values_1m = np.load(SL_VALUES_PATH, mmap_mode="r")
long_tp_1m = np.load(LONG_TP_PATH, mmap_mode="r")
short_tp_1m = np.load(SHORT_TP_PATH, mmap_mode="r")
long_sl_1m = np.load(LONG_SL_PATH, mmap_mode="r")
short_sl_1m = np.load(SHORT_SL_PATH, mmap_mode="r")

print("price_open_time_15m:", price_open_time_15m.shape, price_open_time_15m.dtype)
print("price_close_time_15m:", price_close_time_15m.shape, price_close_time_15m.dtype)
print("price_ohlcv_15m:", price_ohlcv_15m.shape, price_ohlcv_15m.dtype)
for indicator_id, matrix in signal_matrices_15m.items():
    print(indicator_id, matrix.shape, matrix.dtype)
print("tp_values_1m:", tp_values_1m.shape, tp_values_1m.dtype)
print("sl_values_1m:", sl_values_1m.shape, sl_values_1m.dtype)
print("long_tp_1m:", long_tp_1m.shape, long_tp_1m.dtype)
print("short_tp_1m:", short_tp_1m.shape, short_tp_1m.dtype)
print("long_sl_1m:", long_sl_1m.shape, long_sl_1m.dtype)
print("short_sl_1m:", short_sl_1m.shape, short_sl_1m.dtype)


price_open_time_15m: (301866,) int64
price_close_time_15m: (301866,) int64
price_ohlcv_15m: (301866, 5) float32
ma.sma (1176, 301866) int8
momentum.roc (60, 301866) int8
volatility.stddev (48, 301866) int8
tp_values_1m: (5,) float32
sl_values_1m: (5,) float32
long_tp_1m: (5, 4528144) uint32
short_tp_1m: (5, 4528144) uint32
long_sl_1m: (5, 4528144) uint32
short_sl_1m: (5, 4528144) uint32


In [10]:
open_time_index_15m = price_open_time_15m.astype("datetime64[ms]")
close_time_index_15m = price_close_time_15m.astype("datetime64[ms]")
time_mask_15m = (price_open_time_15m >= RUN_TIME_RANGE_START_MS) & (price_open_time_15m <= RUN_TIME_RANGE_END_MS)

price_fields_15m = {
    "open": np.asarray(price_ohlcv_15m[:, 0], dtype=np.float32),
    "high": np.asarray(price_ohlcv_15m[:, 1], dtype=np.float32),
    "low": np.asarray(price_ohlcv_15m[:, 2], dtype=np.float32),
    "close": np.asarray(price_ohlcv_15m[:, 3], dtype=np.float32),
    "volume": np.asarray(price_ohlcv_15m[:, 4], dtype=np.float32),
}
price_fields_15m["hlc3"] = (price_fields_15m["high"] + price_fields_15m["low"] + price_fields_15m["close"]) / 3.0
price_fields_15m["ohlc4"] = (price_fields_15m["open"] + price_fields_15m["high"] + price_fields_15m["low"] + price_fields_15m["close"]) / 4.0

print("15m bars inside run time range:", int(time_mask_15m.sum()))
print("first 15m bar:", np.datetime_as_string(open_time_index_15m[time_mask_15m][0], timezone="UTC"))
print("last 15m bar:", np.datetime_as_string(open_time_index_15m[time_mask_15m][-1], timezone="UTC"))
print("derived price fields:", tuple(price_fields_15m.keys()))


15m bars inside run time range: 296851
first 15m bar: 2017-10-06T23:00:00.000Z
last 15m bar: 2026-03-31T22:45:00.000Z
derived price fields: ('open', 'high', 'low', 'close', 'volume', 'hlc3', 'ohlc4')


In [11]:
def build_source_blocks(matrix, *, sources, param_values):
    """Split one indicator matrix into contiguous source-specific row blocks.

    Parameters:
        matrix: Full signal matrix shaped `(n_rows, n_time)` for one indicator family.
        sources: Ordered source axis values present in the matrix.
        param_values: Ordered parameter values for one source block.

    Returns:
        Dictionary mapping each source name to its contiguous row block.

    Assumptions:
        Rows are ordered by source first and then by parameter value.

    Raises:
        None.

    Side effects:
        None.
    """
    block = len(param_values)
    return {
        source: matrix[idx * block:(idx + 1) * block]
        for idx, source in enumerate(sources)
    }

signal_source_blocks_15m = {
    indicator_id: build_source_blocks(
        signal_matrices_15m[indicator_id],
        sources=meta["sources"],
        param_values=meta["param_values"],
    )
    for indicator_id, meta in EXTRA_INDICATORS_15M.items()
}

for indicator_id, source_map in signal_source_blocks_15m.items():
    print("indicator:", indicator_id)
    for source_name, block in source_map.items():
        print("  ", source_name, block.shape, block.dtype)


indicator: ma.sma
   close (196, 301866) int8
   hlc3 (196, 301866) int8
   ohlc4 (196, 301866) int8
   low (196, 301866) int8
   high (196, 301866) int8
   open (196, 301866) int8
indicator: momentum.roc
   close (10, 301866) int8
   hlc3 (10, 301866) int8
   ohlc4 (10, 301866) int8
   low (10, 301866) int8
   high (10, 301866) int8
   open (10, 301866) int8
indicator: volatility.stddev
   close (8, 301866) int8
   hlc3 (8, 301866) int8
   ohlc4 (8, 301866) int8
   low (8, 301866) int8
   high (8, 301866) int8
   open (8, 301866) int8


In [12]:
sample_indices_15m = np.flatnonzero(time_mask_15m)[:5]
sample_prices_15m = [
    {
        "open_time": np.datetime_as_string(open_time_index_15m[idx], timezone="UTC"),
        "close_time": np.datetime_as_string(close_time_index_15m[idx], timezone="UTC"),
        "open": float(price_fields_15m["open"][idx]),
        "high": float(price_fields_15m["high"][idx]),
        "low": float(price_fields_15m["low"][idx]),
        "close": float(price_fields_15m["close"][idx]),
        "hlc3": float(price_fields_15m["hlc3"][idx]),
        "ohlc4": float(price_fields_15m["ohlc4"][idx]),
        "volume": float(price_fields_15m["volume"][idx]),
    }
    for idx in sample_indices_15m
]
sample_prices_15m


[{'open_time': np.str_('2017-10-06T23:00:00.000Z'),
  'close_time': np.str_('2017-10-06T23:15:00.000Z'),
  'open': 4374.60009765625,
  'high': 4384.990234375,
  'low': 4343.330078125,
  'close': 4384.990234375,
  'hlc3': 4371.103515625,
  'ohlc4': 4371.9775390625,
  'volume': 3.82387113571167},
 {'open_time': np.str_('2017-10-06T23:15:00.000Z'),
  'close_time': np.str_('2017-10-06T23:30:00.000Z'),
  'open': 4383.97998046875,
  'high': 4385.0,
  'low': 4368.3701171875,
  'close': 4385.0,
  'hlc3': 4379.45654296875,
  'ohlc4': 4380.587890625,
  'volume': 1.4752830266952515},
 {'open_time': np.str_('2017-10-06T23:30:00.000Z'),
  'close_time': np.str_('2017-10-06T23:45:00.000Z'),
  'open': 4385.009765625,
  'high': 4399.0,
  'low': 4372.580078125,
  'close': 4390.0,
  'hlc3': 4387.193359375,
  'ohlc4': 4386.6474609375,
  'volume': 2.4674999713897705},
 {'open_time': np.str_('2017-10-06T23:45:00.000Z'),
  'close_time': np.str_('2017-10-07T00:00:00.000Z'),
  'open': 4390.0,
  'high': 4391.68

In [13]:
signal_probe_15m = []
for idx in np.flatnonzero(time_mask_15m)[:8]:
    signal_probe_15m.append({
        "open_time": np.datetime_as_string(open_time_index_15m[idx], timezone="UTC"),
        "ma.sma.close.row_0": int(signal_source_blocks_15m["ma.sma"]["close"][0, idx]),
        "ma.sma.open.row_0": int(signal_source_blocks_15m["ma.sma"]["open"][0, idx]),
        "momentum.roc.low.row_0": int(signal_source_blocks_15m["momentum.roc"]["low"][0, idx]),
        "momentum.roc.ohlc4.row_0": int(signal_source_blocks_15m["momentum.roc"]["ohlc4"][0, idx]),
        "volatility.stddev.high.row_0": int(signal_source_blocks_15m["volatility.stddev"]["high"][0, idx]),
        "volatility.stddev.close.row_0": int(signal_source_blocks_15m["volatility.stddev"]["close"][0, idx]),
    })
signal_probe_15m


[{'open_time': np.str_('2017-10-06T23:00:00.000Z'),
  'ma.sma.close.row_0': 1,
  'ma.sma.open.row_0': 1,
  'momentum.roc.low.row_0': 1,
  'momentum.roc.ohlc4.row_0': 1,
  'volatility.stddev.high.row_0': 1,
  'volatility.stddev.close.row_0': -1},
 {'open_time': np.str_('2017-10-06T23:15:00.000Z'),
  'ma.sma.close.row_0': 1,
  'ma.sma.open.row_0': 1,
  'momentum.roc.low.row_0': 1,
  'momentum.roc.ohlc4.row_0': 1,
  'volatility.stddev.high.row_0': 1,
  'volatility.stddev.close.row_0': 1},
 {'open_time': np.str_('2017-10-06T23:30:00.000Z'),
  'ma.sma.close.row_0': 1,
  'ma.sma.open.row_0': 1,
  'momentum.roc.low.row_0': 1,
  'momentum.roc.ohlc4.row_0': 1,
  'volatility.stddev.high.row_0': 1,
  'volatility.stddev.close.row_0': 1},
 {'open_time': np.str_('2017-10-06T23:45:00.000Z'),
  'ma.sma.close.row_0': -1,
  'ma.sma.open.row_0': 1,
  'momentum.roc.low.row_0': 1,
  'momentum.roc.ohlc4.row_0': 1,
  'volatility.stddev.high.row_0': 1,
  'volatility.stddev.close.row_0': 1},
 {'open_time': np.

## Universal trade-list-first engine experiment on 15m artifacts

This section replaces the earlier brute-force direct-scan experiment with a more universal pipeline inspired by `06_backtest_compute.ipynb`.

Implemented patterns:

- `trade-list-first design`
- `prefilter before exact path`
- `hit-time tables`
- `fast monotone TP/SL kernel`
- `reference-vs-fast self-check`

Practical notes:

- signals use the `15m` artifact matrices for five indicators: `ma.dema`, `ma.ema`, `ma.sma`, `momentum.roc`, `volatility.stddev`
- exact exits use artifact-backed `1m` hit-time tables
- the fast TP/SL kernel runs on the artifact TP/SL grid only, because that is the grid covered by the committed hit-time tables
- exhaustive five-indicator cartesian search is still astronomically large, so the orchestration below is built around bounded row pools, early prefiltering, chunked combo generation, and exact evaluation only on shortlisted candidates


In [14]:
import heapq
import itertools
import math
from math import prod

import numba as nb


def njit_cached(*, parallel: bool = False, fastmath: bool = False, inline: str = "never"):
    """Return a notebook-friendly Numba decorator without filesystem cache dependence.

    Parameters:
        parallel: Whether to enable Numba parallel lowering.
        fastmath: Whether to enable relaxed floating-point optimizations.
        inline: Requested Numba inline policy.

    Returns:
        A decorator that wraps `numba.njit` with `cache=False` for interactive notebook execution.

    Assumptions:
        The notebook executes in an environment where file-backed cache locators may be unavailable.

    Raises:
        None directly; Numba compilation errors propagate from the wrapped function.

    Side effects:
        Triggers Numba compilation on first call of the wrapped function.
    """
    def decorate(func):
        return nb.njit(parallel=parallel, fastmath=fastmath, inline=inline, cache=False)(func)
    return decorate


COMMON_PRICE_SOURCES = ["close", "hlc3", "ohlc4", "low", "high", "open"]
FULL_INDICATORS_15M = {
    "ma.dema": {
        "family": "ma",
        "sources": COMMON_PRICE_SOURCES,
        "param_name": "window",
        "param_values": list(range(5, 201)),
    },
    "ma.ema": {
        "family": "ma",
        "sources": COMMON_PRICE_SOURCES,
        "param_name": "window",
        "param_values": list(range(5, 201)),
    },
    "ma.sma": {
        "family": "ma",
        "sources": COMMON_PRICE_SOURCES,
        "param_name": "window",
        "param_values": list(range(5, 201)),
    },
    "momentum.roc": {
        "family": "momentum",
        "sources": COMMON_PRICE_SOURCES,
        "param_name": "window",
        "param_values": [5, 7, 10, 14, 21, 28, 42, 63, 84, 126],
    },
    "volatility.stddev": {
        "family": "volatility",
        "sources": COMMON_PRICE_SOURCES,
        "param_name": "window",
        "param_values": [10, 14, 20, 28, 42, 56, 84, 126],
    },
}

FULL_SIGNAL_PATHS_15M = {
    indicator_id: {
        "manifest": ARTIFACT_ROOT / "signals" / TIMEFRAME_15M / indicator_id / "manifest.yaml",
        "signals": ARTIFACT_ROOT / "signals" / TIMEFRAME_15M / indicator_id / "signals.i8.npy",
    }
    for indicator_id in FULL_INDICATORS_15M
}

MAPPINGS_15M_DIR = ARTIFACT_ROOT / "mappings" / TIMEFRAME_15M
BAR_OPEN_1M_IDX_15M_PATH = MAPPINGS_15M_DIR / "bar_open_1m_idx.u32.npy"
BAR_CLOSE_1M_IDX_15M_PATH = MAPPINGS_15M_DIR / "bar_close_1m_idx.u32.npy"

PRICE_DIR_1M = ARTIFACT_ROOT / "prices" / "1m"
PRICE_OPEN_TIME_1M_PATH = PRICE_DIR_1M / "open_time.i64.npy"
PRICE_CLOSE_TIME_1M_PATH = PRICE_DIR_1M / "close_time.i64.npy"
PRICE_OHLCV_1M_PATH = PRICE_DIR_1M / "ohlcv.f32.npy"

USE_MMAP = True
PREFILTER_TOP_FRAC = 0.20
PREFILTER_MIN_NONZERO = 200
COMBO_PREFILTER_TOP_FRAC = 0.10
COMBO_MIN_CONFIRM = 30
TIME_CHUNK = 4096
COMBO_CHUNK_SIZE = 2048
TOP_K_DEFAULT = 100
SELF_CHECK_N_DEFAULT = 8
FEE_RATE = 0.0004
CLOSE_ON_END = np.int8(1)
NEG_INF = np.float32(-1e30)
NEG_LARGE = -1.0e30

FIVE_INDICATOR_PATHS = [
    value
    for item in FULL_SIGNAL_PATHS_15M.values()
    for value in item.values()
] + [
    BAR_OPEN_1M_IDX_15M_PATH,
    BAR_CLOSE_1M_IDX_15M_PATH,
    PRICE_OPEN_TIME_1M_PATH,
    PRICE_CLOSE_TIME_1M_PATH,
    PRICE_OHLCV_1M_PATH,
]

for path in FIVE_INDICATOR_PATHS:
    print(f"{path}: exists={path.exists()}")


/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a/signals/15m/ma.dema/manifest.yaml: exists=True
/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a/signals/15m/ma.dema/signals.i8.npy: exists=True
/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a/signals/15m/ma.ema/manifest.yaml: exists=True
/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a/signals/15m/ma.ema/signals.i8.npy: exists=True
/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a/signals/15m/ma.sma/manifest.yaml: exists=True
/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a/signals/15m/ma.sma/signals.i8.npy: exists=True
/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a/signals/15m/momentum.roc/manifest.yaml: exists=True
/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a/signals/15m/momentum.roc/signals.i8.npy: exists=True
/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a/si

In [15]:
full_signal_manifests_15m = {
    indicator_id: yaml.safe_load(paths["manifest"].read_text())
    for indicator_id, paths in FULL_SIGNAL_PATHS_15M.items()
}

full_signal_matrices_15m = {
    indicator_id: np.load(paths["signals"], mmap_mode="r" if USE_MMAP else None)
    for indicator_id, paths in FULL_SIGNAL_PATHS_15M.items()
}

bar_open_1m_idx_15m = np.load(BAR_OPEN_1M_IDX_15M_PATH, mmap_mode="r" if USE_MMAP else None)
bar_close_1m_idx_15m = np.load(BAR_CLOSE_1M_IDX_15M_PATH, mmap_mode="r" if USE_MMAP else None)
price_open_time_1m = np.load(PRICE_OPEN_TIME_1M_PATH, mmap_mode="r" if USE_MMAP else None)
price_close_time_1m = np.load(PRICE_CLOSE_TIME_1M_PATH, mmap_mode="r" if USE_MMAP else None)
price_ohlcv_1m = np.load(PRICE_OHLCV_1M_PATH, mmap_mode="r" if USE_MMAP else None)

run_bar_open_1m_idx_15m = np.asarray(bar_open_1m_idx_15m[time_mask_15m], dtype=np.int32)
run_bar_close_1m_idx_15m = np.asarray(bar_close_1m_idx_15m[time_mask_15m], dtype=np.int32)

price_fields_1m = {
    "open": np.asarray(price_ohlcv_1m[:, 0], dtype=np.float32),
    "high": np.asarray(price_ohlcv_1m[:, 1], dtype=np.float32),
    "low": np.asarray(price_ohlcv_1m[:, 2], dtype=np.float32),
    "close": np.asarray(price_ohlcv_1m[:, 3], dtype=np.float32),
    "volume": np.asarray(price_ohlcv_1m[:, 4], dtype=np.float32),
}

artifact_tp_grid = np.ascontiguousarray(np.asarray(tp_values_1m, dtype=np.float32))
artifact_sl_grid = np.ascontiguousarray(np.asarray(sl_values_1m, dtype=np.float32))
n_tp = int(artifact_tp_grid.size)
n_sl = int(artifact_sl_grid.size)

for hit_name, hit_table in {
    "long_tp": long_tp_1m,
    "long_sl": long_sl_1m,
    "short_tp": short_tp_1m,
    "short_sl": short_sl_1m,
}.items():
    if hit_table.shape[0] > 1 and not np.all(hit_table[1:, :] >= hit_table[:-1, :]):
        raise RuntimeError(f"Hit-time table {hit_name!r} is not monotone across level axis.")

long_tp_eq = np.ascontiguousarray((1.0 + artifact_tp_grid).astype(np.float32))
long_sl_eq = np.ascontiguousarray((1.0 - artifact_sl_grid).astype(np.float32))
short_tp_eq = np.ascontiguousarray((1.0 + artifact_tp_grid).astype(np.float32))
short_sl_eq = np.ascontiguousarray((1.0 - artifact_sl_grid).astype(np.float32))

fee_two_sides = float((1.0 - FEE_RATE) * (1.0 - FEE_RATE))
if fee_two_sides <= 0.0:
    raise ValueError("FEE_RATE produces a non-positive two-sided fee factor.")

log_exec_open_1m = np.zeros(price_fields_1m["open"].shape[0], dtype=np.float64)
positive_open_mask_1m = price_fields_1m["open"] > np.float32(0.0)
if np.any(positive_open_mask_1m):
    log_exec_open_1m[positive_open_mask_1m] = np.log(price_fields_1m["open"][positive_open_mask_1m].astype(np.float64, copy=False))

signal_open_15m_ms = np.asarray(price_open_time_15m[time_mask_15m], dtype=np.int64)
signal_close_15m_ms = np.asarray(price_close_time_15m[time_mask_15m], dtype=np.int64)
signal_close_15m = np.asarray(price_fields_15m["close"][time_mask_15m], dtype=np.float32)
signal_returns_15m = np.ascontiguousarray(((signal_close_15m[1:] / signal_close_15m[:-1]) - 1.0).astype(np.float32))
n_signal_bars = int(signal_close_15m.shape[0])
n_signal_intervals = int(signal_returns_15m.shape[0])

T_exec_limit_1m = np.int32(int(run_bar_close_1m_idx_15m[-1]) + 1)
last_close_1m = float(price_fields_1m["close"][int(T_exec_limit_1m) - 1])
log_last_close_1m = float(math.log(last_close_1m)) if last_close_1m > 0.0 else 0.0
log_fee_two_sides = float(math.log(fee_two_sides))

sig_entry_exec_idx_15m = np.empty(n_signal_bars, dtype=np.int32)
if n_signal_bars > 1:
    sig_entry_exec_idx_15m[:-1] = np.asarray(run_bar_open_1m_idx_15m[1:], dtype=np.int32)
sig_entry_exec_idx_15m[-1] = T_exec_limit_1m

log_fac_tp_long = np.ascontiguousarray(np.log(long_tp_eq.astype(np.float64) * fee_two_sides))
log_fac_sl_long = np.ascontiguousarray(np.log(long_sl_eq.astype(np.float64) * fee_two_sides))
log_fac_tp_short = np.ascontiguousarray(np.log(short_tp_eq.astype(np.float64) * fee_two_sides))
log_fac_sl_short = np.ascontiguousarray(np.log(short_sl_eq.astype(np.float64) * fee_two_sides))

print("signal bars 15m:", n_signal_bars)
print("signal intervals 15m:", n_signal_intervals)
print("execution limit 1m:", int(T_exec_limit_1m))
print("artifact tp grid pct:", (artifact_tp_grid * 100.0).tolist())
print("artifact sl grid pct:", (artifact_sl_grid * 100.0).tolist())
print("fee_two_sides:", fee_two_sides)
for indicator_id, matrix in full_signal_matrices_15m.items():
    print(indicator_id, matrix.shape, matrix.dtype)


signal bars 15m: 296851
signal intervals 15m: 296850
execution limit 1m: 4525448
artifact tp grid pct: [0.5, 1.0, 1.5, 2.0, 3.0]
artifact sl grid pct: [0.5, 1.0, 1.5, 2.0, 3.0]
fee_two_sides: 0.9992001600000001
ma.dema (1176, 301866) int8
ma.ema (1176, 301866) int8
ma.sma (1176, 301866) int8
momentum.roc (60, 301866) int8
volatility.stddev (48, 301866) int8


In [16]:
def build_row_catalog(*, indicator_id, sources, param_name, param_values):
    """Build row metadata for one artifact-backed indicator matrix.

    Parameters:
        indicator_id: Indicator id matching the artifact directory name.
        sources: Ordered source axis for this artifact matrix.
        param_name: Human-readable parameter axis name.
        param_values: Ordered parameter values inside each source block.

    Returns:
        A list of dictionaries with `row_id`, `source`, and parameter value for every artifact row.

    Assumptions:
        The artifact layout is source-major and parameter-minor inside each source block.

    Raises:
        None.

    Side effects:
        None.
    """
    rows = []
    block = len(param_values)
    for source_idx, source_name in enumerate(sources):
        base = source_idx * block
        for offset, param_value in enumerate(param_values):
            rows.append({
                "indicator_id": indicator_id,
                "row_id": base + offset,
                "source": source_name,
                param_name: int(param_value),
            })
    return rows


def row_ids_for_sources(*, indicator_id, source_names):
    """Return original artifact row ids for the requested source blocks.

    Parameters:
        indicator_id: Indicator id present in `FULL_INDICATORS_15M`.
        source_names: Iterable of source names to select.

    Returns:
        An `int32` array with the original artifact row ids for the requested sources.

    Assumptions:
        Each source occupies one contiguous block of rows.

    Raises:
        KeyError: If the requested source is not present for the indicator.

    Side effects:
        None.
    """
    meta = FULL_INDICATORS_15M[indicator_id]
    param_count = len(meta["param_values"])
    source_to_index = {name: idx for idx, name in enumerate(meta["sources"])}
    row_ids = []
    for source_name in source_names:
        source_idx = source_to_index[source_name]
        start = source_idx * param_count
        row_ids.extend(range(start, start + param_count))
    return np.asarray(row_ids, dtype=np.int32)


def extract_signal_rows(*, indicator_id, row_ids):
    """Load a bounded set of 15m signal rows for one indicator into a contiguous matrix.

    Parameters:
        indicator_id: Indicator id present in `full_signal_matrices_15m`.
        row_ids: Original artifact row ids to extract.

    Returns:
        Int8 matrix with shape `(len(row_ids), n_signal_bars)` over the run time range.

    Assumptions:
        `time_mask_15m` already bounds the active experiment range.

    Raises:
        ValueError: If the row id list is empty.

    Side effects:
        Copies the selected memmap rows into a contiguous in-memory matrix.
    """
    row_ids = np.asarray(row_ids, dtype=np.int32)
    if row_ids.size == 0:
        raise ValueError(f"Empty row selection for {indicator_id!r}.")
    return np.ascontiguousarray(np.asarray(full_signal_matrices_15m[indicator_id][row_ids][:, time_mask_15m], dtype=np.int8))


five_indicator_row_catalogs_15m = {
    indicator_id: build_row_catalog(
        indicator_id=indicator_id,
        sources=meta["sources"],
        param_name=meta["param_name"],
        param_values=meta["param_values"],
    )
    for indicator_id, meta in FULL_INDICATORS_15M.items()
}

five_indicator_row_counts = {
    indicator_id: int(matrix.shape[0])
    for indicator_id, matrix in full_signal_matrices_15m.items()
}
full_variant_count_5 = prod(five_indicator_row_counts.values())
full_variant_count_with_artifact_grid = full_variant_count_5 * n_tp * n_sl

print("five indicator row counts:", five_indicator_row_counts)
print("plain five-indicator combinations:", full_variant_count_5)
print("five-indicator combinations with artifact TP/SL grid:", full_variant_count_with_artifact_grid)
print("sample ma.ema rows:", five_indicator_row_catalogs_15m["ma.ema"][:3])


five indicator row counts: {'ma.dema': 1176, 'ma.ema': 1176, 'ma.sma': 1176, 'momentum.roc': 60, 'volatility.stddev': 48}
plain five-indicator combinations: 4683973754880
five-indicator combinations with artifact TP/SL grid: 117099343872000
sample ma.ema rows: [{'indicator_id': 'ma.ema', 'row_id': 0, 'source': 'close', 'window': 5}, {'indicator_id': 'ma.ema', 'row_id': 1, 'source': 'close', 'window': 6}, {'indicator_id': 'ma.ema', 'row_id': 2, 'source': 'close', 'window': 7}]


In [17]:
def topk_fraction_idx(score: np.ndarray, frac: float) -> np.ndarray:
    """Select top indices by fraction from a one-dimensional score vector.

    Parameters:
        score: One-dimensional score array.
        frac: Fraction in `(0, 1]` describing how many elements to keep.

    Returns:
        Int32 indices of the kept elements.

    Assumptions:
        The input contains at least one finite element.

    Raises:
        ValueError: If `frac` is outside `(0, 1]`.

    Side effects:
        None.
    """
    if not (0.0 < frac <= 1.0):
        raise ValueError(f"frac must be in (0, 1], got {frac!r}")
    n = int(score.shape[0])
    k = max(1, int(math.ceil(n * frac)))
    if k >= n:
        return np.arange(n, dtype=np.int32)
    idx = np.argpartition(score, n - k)[n - k:]
    return np.sort(idx.astype(np.int32))


def single_score_chunked(sig_T_i8: np.ndarray, ret_f32: np.ndarray, chunk: int) -> np.ndarray:
    """Compute a chunked dot-product proxy score for one indicator family.

    Parameters:
        sig_T_i8: Int8 signal matrix shaped `(n_rows, n_intervals)`.
        ret_f32: Float32 return vector shaped `(n_intervals,)`.
        chunk: Chunk length along the time axis.

    Returns:
        Float32 proxy score per row.

    Assumptions:
        The signal matrix is already aligned to the return intervals.

    Raises:
        ValueError: If matrix and return lengths disagree.

    Side effects:
        None.
    """
    if sig_T_i8.shape[1] != ret_f32.shape[0]:
        raise ValueError("Signal matrix and return vector must share the same interval length.")
    n_rows, n_intervals = sig_T_i8.shape
    out = np.zeros(n_rows, dtype=np.float32)
    for t0 in range(0, n_intervals, chunk):
        t1 = min(t0 + chunk, n_intervals)
        out += sig_T_i8[:, t0:t1].astype(np.float32) @ ret_f32[t0:t1]
    return out


def prefilter_indicator_rows(*, trade_T: np.ndarray, indicator_id: str, row_ids: np.ndarray, top_frac: float, min_nonzero: int, fee_rate: float, time_chunk: int):
    """Apply a cheap single-indicator prefilter before exact combo evaluation.

    Parameters:
        trade_T: Int8 signal matrix shaped `(n_rows, n_signal_bars)`.
        indicator_id: Indicator id for diagnostics.
        row_ids: Original artifact row ids aligned to `trade_T` rows.
        top_frac: Fraction of rows to retain after scoring.
        min_nonzero: Minimum number of non-zero signals required to keep a row.
        fee_rate: Per-side fee used as a rough penalty term in the proxy score.
        time_chunk: Chunk size for the proxy score computation.

    Returns:
        Dictionary with filtered row ids, filtered trade matrix, eval matrix, adjusted scores, and non-zero counts.

    Assumptions:
        The exact path will trade on the next bar, so the proxy evaluation uses `trade_T[:, :-1]`.

    Raises:
        ValueError: If no candidate survives the non-zero filter.

    Side effects:
        None.
    """
    row_ids = np.asarray(row_ids, dtype=np.int32)
    if trade_T.shape[0] != row_ids.shape[0]:
        raise ValueError(f"Row id alignment mismatch for {indicator_id!r}.")

    eval_T = np.ascontiguousarray(trade_T[:, :n_signal_intervals])
    nonzero = (eval_T != 0).sum(axis=1).astype(np.int32)
    proxy = single_score_chunked(eval_T, signal_returns_15m, chunk=time_chunk)
    adjusted = proxy - (fee_rate * nonzero.astype(np.float32))
    valid = nonzero >= int(min_nonzero)
    if not np.any(valid):
        raise ValueError(f"No rows survive min_nonzero={min_nonzero} for {indicator_id!r}.")

    valid_idx = np.flatnonzero(valid)
    keep_from_valid = topk_fraction_idx(adjusted[valid_idx], top_frac)
    keep_idx = np.sort(valid_idx[keep_from_valid].astype(np.int32))

    return {
        "indicator_id": indicator_id,
        "filtered_row_ids": row_ids[keep_idx],
        "trade_T": np.ascontiguousarray(trade_T[keep_idx]),
        "eval_T": np.ascontiguousarray(eval_T[keep_idx]),
        "score_adj": np.ascontiguousarray(adjusted[keep_idx]),
        "nonzero": np.ascontiguousarray(nonzero[keep_idx]),
    }


def prepare_indicator_pool(*, indicator_id: str, row_ids: np.ndarray | None = None, top_frac: float = PREFILTER_TOP_FRAC, min_nonzero: int = PREFILTER_MIN_NONZERO, fee_rate: float = FEE_RATE, time_chunk: int = TIME_CHUNK):
    """Load and prefilter one indicator pool for the exact five-indicator engine.

    Parameters:
        indicator_id: Indicator id to prepare.
        row_ids: Optional original artifact row ids. When omitted, all rows are used.
        top_frac: Fraction of rows to keep after proxy scoring.
        min_nonzero: Minimum count of non-zero signals required for a row.
        fee_rate: Fee penalty used inside the proxy prefilter.
        time_chunk: Chunk size for proxy scoring.

    Returns:
        Dictionary returned by `prefilter_indicator_rows` for this indicator.

    Assumptions:
        Row metadata is available in `five_indicator_row_catalogs_15m`.

    Raises:
        ValueError: If the initial row selection is empty.

    Side effects:
        Loads the selected signal rows into memory.
    """
    if row_ids is None:
        row_ids = np.arange(five_indicator_row_counts[indicator_id], dtype=np.int32)
    row_ids = np.asarray(row_ids, dtype=np.int32)
    if row_ids.size == 0:
        raise ValueError(f"Empty requested pool for {indicator_id!r}.")
    trade_T = extract_signal_rows(indicator_id=indicator_id, row_ids=row_ids)
    return prefilter_indicator_rows(
        trade_T=trade_T,
        indicator_id=indicator_id,
        row_ids=row_ids,
        top_frac=top_frac,
        min_nonzero=min_nonzero,
        fee_rate=fee_rate,
        time_chunk=time_chunk,
    )


In [18]:
@njit_cached(inline="always")
def consensus_dir5(dema_value: np.int8, ema_value: np.int8, sma_value: np.int8, roc_value: np.int8, std_value: np.int8) -> np.int8:
    """Resolve the five-indicator consensus direction for one signal bar.

    Parameters:
        dema_value: DEMA signal on one bar.
        ema_value: EMA signal on one bar.
        sma_value: SMA signal on one bar.
        roc_value: ROC signal on one bar.
        std_value: Stddev signal on one bar.

    Returns:
        `1` for unanimous long, `-1` for unanimous short, `0` otherwise.

    Assumptions:
        All inputs are already normalized to `{-1, 0, 1}`.

    Raises:
        None.

    Side effects:
        None.
    """
    if dema_value == 1 and ema_value == 1 and sma_value == 1 and roc_value == 1 and std_value == 1:
        return np.int8(1)
    if dema_value == -1 and ema_value == -1 and sma_value == -1 and roc_value == -1 and std_value == -1:
        return np.int8(-1)
    return np.int8(0)


@njit_cached()
def build_trade_list_for_five_rows(
    dema_sig_row: np.ndarray,
    ema_sig_row: np.ndarray,
    sma_sig_row: np.ndarray,
    roc_sig_row: np.ndarray,
    std_sig_row: np.ndarray,
    sig_entry_exec_idx: np.ndarray,
    T_exec: np.int32,
    out_entry_exec_idx: np.ndarray,
    out_dir: np.ndarray,
    out_sig_exit_exec_idx: np.ndarray,
) -> np.int32:
    """Build a compact trade list for one five-indicator consensus strategy.

    Parameters:
        dema_sig_row: DEMA signal row over the active 15m run range.
        ema_sig_row: EMA signal row over the active 15m run range.
        sma_sig_row: SMA signal row over the active 15m run range.
        roc_sig_row: ROC signal row over the active 15m run range.
        std_sig_row: Stddev signal row over the active 15m run range.
        sig_entry_exec_idx: Mapping from each 15m signal bar to the next 1m execution entry index.
        T_exec: Exclusive 1m execution limit for the run range.
        out_entry_exec_idx: Preallocated output for trade entry indices.
        out_dir: Preallocated output for trade directions.
        out_sig_exit_exec_idx: Preallocated output for signal-exit indices.

    Returns:
        Number of trades written into the output buffers, or `-1` when capacity is insufficient.

    Assumptions:
        Repeated confirmations in the same direction while already in a position are ignored.

    Raises:
        None directly; buffer overflow is reported via `-1`.

    Side effects:
        Mutates the provided output arrays in place.
    """
    n_sig = dema_sig_row.shape[0]
    n_trades = np.int32(0)
    current_dir = np.int8(0)
    current_entry = np.int32(0)

    for t in range(n_sig):
        dirn = consensus_dir5(
            dema_sig_row[t],
            ema_sig_row[t],
            sma_sig_row[t],
            roc_sig_row[t],
            std_sig_row[t],
        )
        if dirn == 0:
            continue

        entry_exec = sig_entry_exec_idx[t]
        if entry_exec >= T_exec:
            break

        if current_dir == 0:
            current_dir = dirn
            current_entry = np.int32(entry_exec)
            continue

        if dirn == current_dir:
            continue

        if n_trades >= out_entry_exec_idx.shape[0]:
            return np.int32(-1)

        out_entry_exec_idx[n_trades] = current_entry
        out_dir[n_trades] = current_dir
        out_sig_exit_exec_idx[n_trades] = np.int32(entry_exec)
        n_trades += 1
        current_dir = dirn
        current_entry = np.int32(entry_exec)

    if current_dir != 0:
        if n_trades >= out_entry_exec_idx.shape[0]:
            return np.int32(-1)
        out_entry_exec_idx[n_trades] = current_entry
        out_dir[n_trades] = current_dir
        out_sig_exit_exec_idx[n_trades] = T_exec
        n_trades += 1

    return n_trades


@njit_cached(parallel=True)
def count_trades_for_five_combos(
    combo_dema_idx: np.ndarray,
    combo_ema_idx: np.ndarray,
    combo_sma_idx: np.ndarray,
    combo_roc_idx: np.ndarray,
    combo_std_idx: np.ndarray,
    dema_trade_T: np.ndarray,
    ema_trade_T: np.ndarray,
    sma_trade_T: np.ndarray,
    roc_trade_T: np.ndarray,
    std_trade_T: np.ndarray,
    sig_entry_exec_idx: np.ndarray,
    T_exec: np.int32,
    out_trade_counts: np.ndarray,
) -> None:
    """Count compressed trades for a chunk of five-indicator combinations.

    Parameters:
        combo_*_idx: Row indices into the filtered trade matrices for each indicator.
        *_trade_T: Filtered trade matrices shaped `(n_rows, n_signal_bars)`.
        sig_entry_exec_idx: 15m signal-bar to 1m execution-entry mapping.
        T_exec: Exclusive 1m execution limit for the run range.
        out_trade_counts: Output vector for trade counts per combination.

    Returns:
        None.

    Assumptions:
        All combo index arrays share the same length.

    Raises:
        None.

    Side effects:
        Writes counts into `out_trade_counts`.
    """
    K = combo_dema_idx.shape[0]
    n_sig = dema_trade_T.shape[1]

    for k in nb.prange(K):
        current_dir = np.int8(0)
        n_trades = np.int32(0)
        di = combo_dema_idx[k]
        ei = combo_ema_idx[k]
        si = combo_sma_idx[k]
        ri = combo_roc_idx[k]
        vi = combo_std_idx[k]

        for t in range(n_sig):
            dirn = consensus_dir5(
                dema_trade_T[di, t],
                ema_trade_T[ei, t],
                sma_trade_T[si, t],
                roc_trade_T[ri, t],
                std_trade_T[vi, t],
            )
            if dirn == 0:
                continue
            if sig_entry_exec_idx[t] >= T_exec:
                break
            if current_dir == 0:
                current_dir = dirn
                continue
            if dirn != current_dir:
                n_trades += 1
                current_dir = dirn

        if current_dir != 0:
            n_trades += 1
        out_trade_counts[k] = n_trades


@njit_cached(inline="always")
def evaluate_trade_factor_hit_tables(
    dirn: np.int8,
    entry_exec: np.int32,
    sig_exit_exec: np.int32,
    tp_i: np.int32,
    sl_i: np.int32,
    exec_open: np.ndarray,
    last_close: float,
    T_exec: np.int32,
    close_on_end: np.int8,
    hit_long_tp: np.ndarray,
    hit_long_sl: np.ndarray,
    hit_short_tp: np.ndarray,
    hit_short_sl: np.ndarray,
    long_tp_eq: np.ndarray,
    long_sl_eq: np.ndarray,
    short_tp_eq: np.ndarray,
    short_sl_eq: np.ndarray,
) -> tuple[float, np.int32, np.int8]:
    """Resolve one trade exit from artifact hit-time tables and return the gross factor before fees.

    Parameters:
        dirn: Trade direction (`1` long, `-1` short).
        entry_exec: Absolute 1m entry index.
        sig_exit_exec: Absolute 1m signal-exit index, or `T_exec` for an open final trade.
        tp_i: TP level index.
        sl_i: SL level index.
        exec_open: Full 1m open series.
        last_close: Close price of the last 1m bar inside the active run range.
        T_exec: Exclusive 1m execution limit for the run range.
        close_on_end: Whether an open trade should close on the final bar.
        hit_*: Absolute 1m hit-time lookup tables.
        *_eq: Gross TP/SL factors for each grid level.

    Returns:
        Tuple `(gross_factor, exit_exec_idx, closed_flag)`.

    Assumptions:
        TP/SL lookup starts from `entry_exec + 1` to avoid entry-bar lookahead.

    Raises:
        None.

    Side effects:
        None.
    """
    entry_open = float(exec_open[entry_exec])
    if entry_open <= 0.0:
        return 1.0, entry_exec, np.int8(0)

    lookup_exec = np.int32(entry_exec + 1)
    ttp = T_exec
    tsl = T_exec
    tp_pf = 1.0
    sl_pf = 1.0

    if lookup_exec < T_exec:
        if dirn == 1:
            ttp = np.int32(hit_long_tp[tp_i, lookup_exec])
            tsl = np.int32(hit_long_sl[sl_i, lookup_exec])
            tp_pf = float(long_tp_eq[tp_i])
            sl_pf = float(long_sl_eq[sl_i])
        else:
            ttp = np.int32(hit_short_tp[tp_i, lookup_exec])
            tsl = np.int32(hit_short_sl[sl_i, lookup_exec])
            tp_pf = float(short_tp_eq[tp_i])
            sl_pf = float(short_sl_eq[sl_i])

    if tsl <= ttp:
        tp_sl_exec = tsl
        tp_sl_pf = sl_pf
    else:
        tp_sl_exec = ttp
        tp_sl_pf = tp_pf

    if sig_exit_exec < T_exec and sig_exit_exec <= tp_sl_exec:
        exit_open = float(exec_open[sig_exit_exec])
        if exit_open > 0.0:
            if dirn == 1:
                pf = exit_open / entry_open
            else:
                ratio = exit_open / entry_open
                pf = 2.0 - ratio
                if pf <= 0.0:
                    pf = 0.0
        else:
            pf = 1.0
        return pf, sig_exit_exec, np.int8(1)

    if tp_sl_exec < T_exec:
        return tp_sl_pf, tp_sl_exec, np.int8(1)

    if close_on_end == 1 and T_exec > 0:
        if last_close > 0.0:
            if dirn == 1:
                pf = last_close / entry_open
            else:
                ratio = last_close / entry_open
                pf = 2.0 - ratio
                if pf <= 0.0:
                    pf = 0.0
        else:
            pf = 1.0
        return pf, np.int32(T_exec - 1), np.int8(1)

    return 1.0, entry_exec, np.int8(0)


In [19]:
@njit_cached(inline="always")
def add_row_range(row_diff: np.ndarray, row_i: np.int32, col_start: np.int32, col_stop: np.int32, value: float) -> None:
    """Add a value to a contiguous row segment in a row-wise difference buffer.

    Parameters:
        row_diff: Difference buffer shaped `(n_tp, n_sl + 1)`.
        row_i: TP row index to update.
        col_start: Inclusive SL start index.
        col_stop: Exclusive SL stop index.
        value: Log contribution to add.

    Returns:
        None.

    Assumptions:
        Indices are already clamped to valid bounds.

    Raises:
        None.

    Side effects:
        Mutates `row_diff` in place.
    """
    if col_start < col_stop:
        row_diff[row_i, col_start] += value
        row_diff[row_i, col_stop] -= value


@njit_cached(inline="always")
def add_col_range(col_diff: np.ndarray, row_start: np.int32, row_stop: np.int32, col_i: np.int32, value: float) -> None:
    """Add a value to a contiguous column segment in a column-wise difference buffer.

    Parameters:
        col_diff: Difference buffer shaped `(n_tp + 1, n_sl)`.
        row_start: Inclusive TP start index.
        row_stop: Exclusive TP stop index.
        col_i: SL column index to update.
        value: Log contribution to add.

    Returns:
        None.

    Assumptions:
        Indices are already clamped to valid bounds.

    Raises:
        None.

    Side effects:
        Mutates `col_diff` in place.
    """
    if row_start < row_stop:
        col_diff[row_start, col_i] += value
        col_diff[row_stop, col_i] -= value


@njit_cached(inline="always")
def add_rect(rect_diff: np.ndarray, row_start: np.int32, col_start: np.int32, row_stop: np.int32, col_stop: np.int32, value: float) -> None:
    """Add a value to an axis-aligned rectangle in a 2D difference buffer.

    Parameters:
        rect_diff: Difference buffer shaped `(n_tp + 1, n_sl + 1)`.
        row_start: Inclusive TP start index.
        col_start: Inclusive SL start index.
        row_stop: Exclusive TP stop index.
        col_stop: Exclusive SL stop index.
        value: Log contribution to add.

    Returns:
        None.

    Assumptions:
        Indices are already clamped to valid bounds.

    Raises:
        None.

    Side effects:
        Mutates `rect_diff` in place.
    """
    if row_start < row_stop and col_start < col_stop:
        rect_diff[row_start, col_start] += value
        rect_diff[row_stop, col_start] -= value
        rect_diff[row_start, col_stop] -= value
        rect_diff[row_stop, col_stop] += value


@njit_cached(inline="always")
def lower_bound_ge_hit(hit_table: np.ndarray, start_exec: np.int32, n_levels: np.int32, target: np.int32) -> np.int32:
    """Find the first level whose hit time is greater than or equal to a target bar.

    Parameters:
        hit_table: Monotone hit-time table shaped `(n_levels, T_exec_full)`.
        start_exec: Absolute 1m start index for lookup.
        n_levels: Number of levels in the level axis.
        target: Absolute 1m target index.

    Returns:
        The first level index with hit time `>= target`, or `n_levels` if no such level exists.

    Assumptions:
        Hit times are non-decreasing across the level axis for the chosen start column.

    Raises:
        None.

    Side effects:
        None.
    """
    lo = np.int32(0)
    hi = np.int32(n_levels)
    while lo < hi:
        mid = np.int32((lo + hi) // 2)
        if np.int32(hit_table[mid, start_exec]) >= target:
            hi = mid
        else:
            lo = np.int32(mid + 1)
    return lo


@njit_cached(inline="always")
def first_equal_hit(hit_table: np.ndarray, start_exec: np.int32, n_levels: np.int32, target: np.int32) -> np.int32:
    """Find the first level whose hit time exactly matches the target index.

    Parameters:
        hit_table: Monotone hit-time table shaped `(n_levels, T_exec_full)`.
        start_exec: Absolute 1m start index for lookup.
        n_levels: Number of levels in the level axis.
        target: Absolute 1m target index.

    Returns:
        The first matching level index, or `n_levels` if no exact match exists.

    Assumptions:
        Hit times are non-decreasing across the level axis for the chosen start column.

    Raises:
        None.

    Side effects:
        None.
    """
    idx = lower_bound_ge_hit(hit_table, start_exec, n_levels, target)
    if idx < n_levels and np.int32(hit_table[idx, start_exec]) == target:
        return idx
    return np.int32(n_levels)


@njit_cached(parallel=True, fastmath=True)
def evaluate_best_tp_sl_trade_list_slow_five(
    combo_dema_idx: np.ndarray,
    combo_ema_idx: np.ndarray,
    combo_sma_idx: np.ndarray,
    combo_roc_idx: np.ndarray,
    combo_std_idx: np.ndarray,
    dema_trade_T: np.ndarray,
    ema_trade_T: np.ndarray,
    sma_trade_T: np.ndarray,
    roc_trade_T: np.ndarray,
    std_trade_T: np.ndarray,
    sig_entry_exec_idx: np.ndarray,
    exec_open_1m: np.ndarray,
    last_close_1m: float,
    T_exec: np.int32,
    hit_long_tp: np.ndarray,
    hit_long_sl: np.ndarray,
    hit_short_tp: np.ndarray,
    hit_short_sl: np.ndarray,
    long_tp_eq: np.ndarray,
    long_sl_eq: np.ndarray,
    short_tp_eq: np.ndarray,
    short_sl_eq: np.ndarray,
    fee_two_sides: float,
    close_on_end: np.int8,
    out_best_tp_idx: np.ndarray,
    out_best_sl_idx: np.ndarray,
    out_best_ret: np.ndarray,
    out_trade_counts: np.ndarray,
) -> None:
    """Reference TP/SL grid search for five-indicator combos by explicit cell replay.

    Parameters:
        combo_*_idx: Combo row indices into the filtered trade matrices.
        *_trade_T: Filtered trade matrices for the five indicators.
        sig_entry_exec_idx: 15m signal-bar to 1m execution-entry mapping.
        exec_open_1m: Full 1m open series.
        last_close_1m: Close price of the last 1m bar inside the active run range.
        T_exec: Exclusive 1m execution limit for the run range.
        hit_*: Artifact hit-time tables.
        *_eq: TP/SL gross factors.
        fee_two_sides: Two-sided fee factor.
        close_on_end: Whether to close a final open trade at the end.
        out_best_*: Output arrays for the best grid cell and total return.
        out_trade_counts: Output trade-count vector.

    Returns:
        None.

    Assumptions:
        This kernel is used only for a small self-check subset because its complexity is `O(K * trade_count * n_tp * n_sl)`.

    Raises:
        None.

    Side effects:
        Writes results into the provided output arrays.
    """
    K = combo_dema_idx.shape[0]
    n_tp_local = hit_long_tp.shape[0]
    n_sl_local = hit_long_sl.shape[0]
    n_sig = dema_trade_T.shape[1]

    for k in nb.prange(K):
        di = combo_dema_idx[k]
        ei = combo_ema_idx[k]
        si = combo_sma_idx[k]
        ri = combo_roc_idx[k]
        vi = combo_std_idx[k]

        entry_arr = np.empty(n_sig, dtype=np.int32)
        dir_arr = np.empty(n_sig, dtype=np.int8)
        sig_exit_arr = np.empty(n_sig, dtype=np.int32)

        n_trades = build_trade_list_for_five_rows(
            dema_trade_T[di],
            ema_trade_T[ei],
            sma_trade_T[si],
            roc_trade_T[ri],
            std_trade_T[vi],
            sig_entry_exec_idx,
            T_exec,
            entry_arr,
            dir_arr,
            sig_exit_arr,
        )
        out_trade_counts[k] = n_trades

        if n_trades <= 0:
            out_best_tp_idx[k] = np.int32(0)
            out_best_sl_idx[k] = np.int32(0)
            out_best_ret[k] = np.float32(0.0)
            continue

        best_eq = -1.0
        best_tp = np.int32(0)
        best_sl = np.int32(0)

        for tp_i in range(n_tp_local):
            for sl_i in range(n_sl_local):
                eq = 1.0
                for tr in range(n_trades):
                    pf, _, closed = evaluate_trade_factor_hit_tables(
                        dir_arr[tr],
                        entry_arr[tr],
                        sig_exit_arr[tr],
                        np.int32(tp_i),
                        np.int32(sl_i),
                        exec_open_1m,
                        last_close_1m,
                        T_exec,
                        close_on_end,
                        hit_long_tp,
                        hit_long_sl,
                        hit_short_tp,
                        hit_short_sl,
                        long_tp_eq,
                        long_sl_eq,
                        short_tp_eq,
                        short_sl_eq,
                    )
                    if closed == 1:
                        eq *= fee_two_sides * pf
                if eq > best_eq:
                    best_eq = eq
                    best_tp = np.int32(tp_i)
                    best_sl = np.int32(sl_i)

        out_best_tp_idx[k] = best_tp
        out_best_sl_idx[k] = best_sl
        out_best_ret[k] = np.float32(best_eq - 1.0)


@njit_cached(parallel=True, fastmath=True)
def evaluate_best_tp_sl_trade_list_fast_monotone_five(
    combo_dema_idx: np.ndarray,
    combo_ema_idx: np.ndarray,
    combo_sma_idx: np.ndarray,
    combo_roc_idx: np.ndarray,
    combo_std_idx: np.ndarray,
    dema_trade_T: np.ndarray,
    ema_trade_T: np.ndarray,
    sma_trade_T: np.ndarray,
    roc_trade_T: np.ndarray,
    std_trade_T: np.ndarray,
    sig_entry_exec_idx: np.ndarray,
    exec_open_1m: np.ndarray,
    last_close_1m: float,
    T_exec: np.int32,
    hit_long_tp: np.ndarray,
    hit_long_sl: np.ndarray,
    hit_short_tp: np.ndarray,
    hit_short_sl: np.ndarray,
    long_tp_eq: np.ndarray,
    long_sl_eq: np.ndarray,
    short_tp_eq: np.ndarray,
    short_sl_eq: np.ndarray,
    trade_counts: np.ndarray,
    log_fac_tp_long: np.ndarray,
    log_fac_sl_long: np.ndarray,
    log_fac_tp_short: np.ndarray,
    log_fac_sl_short: np.ndarray,
    log_fee_two_sides: float,
    log_exec_open_1m: np.ndarray,
    log_last_close_1m: float,
    fee_two_sides: float,
    close_on_end: np.int8,
    out_best_tp_idx: np.ndarray,
    out_best_sl_idx: np.ndarray,
    out_best_ret: np.ndarray,
    out_trade_counts: np.ndarray,
) -> None:
    """Find the best TP/SL cell per combo with monotone hit-time decomposition.

    Parameters:
        combo_*_idx: Combo row indices into the filtered trade matrices.
        *_trade_T: Filtered trade matrices for the five indicators.
        sig_entry_exec_idx: 15m signal-bar to 1m execution-entry mapping.
        exec_open_1m: Full 1m open series.
        last_close_1m: Close price of the last 1m bar inside the active run range.
        T_exec: Exclusive 1m execution limit for the run range.
        hit_*: Artifact hit-time tables.
        *_eq: TP/SL gross factors.
        trade_counts: Precomputed compact trade counts per combination.
        log_fac_*: Log TP/SL contributions including fees.
        log_fee_two_sides: Log of the two-sided fee factor.
        log_exec_open_1m: Log-open vector on the full 1m series.
        log_last_close_1m: Log of `last_close_1m` when positive, otherwise zero.
        fee_two_sides: Two-sided fee factor.
        close_on_end: Whether a final open trade closes at the run end.
        out_best_*: Output arrays for best TP/SL and return.
        out_trade_counts: Output trade-count vector.

    Returns:
        None.

    Assumptions:
        Hit-time tables are monotone non-decreasing across the level axis.

    Raises:
        None.

    Side effects:
        Writes results into the provided output arrays.
    """
    K = combo_dema_idx.shape[0]
    n_tp_local = hit_long_tp.shape[0]
    n_sl_local = hit_long_sl.shape[0]

    for k in nb.prange(K):
        alloc_n = np.int32(trade_counts[k])
        if alloc_n <= 0:
            out_trade_counts[k] = np.int32(0)
            out_best_tp_idx[k] = np.int32(0)
            out_best_sl_idx[k] = np.int32(0)
            out_best_ret[k] = np.float32(0.0)
            continue

        di = combo_dema_idx[k]
        ei = combo_ema_idx[k]
        si = combo_sma_idx[k]
        ri = combo_roc_idx[k]
        vi = combo_std_idx[k]

        entry_arr = np.empty(alloc_n, dtype=np.int32)
        dir_arr = np.empty(alloc_n, dtype=np.int8)
        sig_exit_arr = np.empty(alloc_n, dtype=np.int32)

        n_trades = build_trade_list_for_five_rows(
            dema_trade_T[di],
            ema_trade_T[ei],
            sma_trade_T[si],
            roc_trade_T[ri],
            std_trade_T[vi],
            sig_entry_exec_idx,
            T_exec,
            entry_arr,
            dir_arr,
            sig_exit_arr,
        )
        out_trade_counts[k] = n_trades
        if n_trades <= 0:
            out_best_tp_idx[k] = np.int32(0)
            out_best_sl_idx[k] = np.int32(0)
            out_best_ret[k] = np.float32(0.0)
            continue

        row_diff = np.zeros((n_tp_local, n_sl_local + 1), dtype=np.float64)
        col_diff = np.zeros((n_tp_local + 1, n_sl_local), dtype=np.float64)
        rect_diff = np.zeros((n_tp_local + 1, n_sl_local + 1), dtype=np.float64)

        for tr in range(n_trades):
            dirn = dir_arr[tr]
            entry_exec = entry_arr[tr]
            sig_exit_exec = sig_exit_arr[tr]

            entry_open = float(exec_open_1m[entry_exec])
            if entry_open <= 0.0:
                continue
            entry_log = float(log_exec_open_1m[entry_exec])
            start = np.int32(entry_exec + 1)

            if dirn == 1:
                hit_tp = hit_long_tp
                hit_sl = hit_long_sl
                log_tp_arr = log_fac_tp_long
                log_sl_arr = log_fac_sl_long
            else:
                hit_tp = hit_short_tp
                hit_sl = hit_short_sl
                log_tp_arr = log_fac_tp_short
                log_sl_arr = log_fac_sl_short

            if start >= T_exec:
                if sig_exit_exec < T_exec:
                    exit_open = float(exec_open_1m[sig_exit_exec])
                    contrib = log_fee_two_sides
                    if exit_open > 0.0:
                        if dirn == 1:
                            contrib += float(log_exec_open_1m[sig_exit_exec]) - entry_log
                        else:
                            ratio = exit_open / entry_open
                            if ratio >= 2.0:
                                contrib = NEG_LARGE
                            else:
                                contrib += math.log(2.0 - ratio)
                    add_rect(rect_diff, np.int32(0), np.int32(0), np.int32(n_tp_local), np.int32(n_sl_local), contrib)
                elif close_on_end == 1 and T_exec > 0:
                    contrib = log_fee_two_sides
                    if last_close_1m > 0.0:
                        if dirn == 1:
                            contrib += log_last_close_1m - entry_log
                        else:
                            ratio = last_close_1m / entry_open
                            if ratio >= 2.0:
                                contrib = NEG_LARGE
                            else:
                                contrib += math.log(2.0 - ratio)
                    add_rect(rect_diff, np.int32(0), np.int32(0), np.int32(n_tp_local), np.int32(n_sl_local), contrib)
                continue

            if sig_exit_exec < T_exec:
                t_sig = np.int32(sig_exit_exec)
                i_sig = lower_bound_ge_hit(hit_tp, start, np.int32(n_tp_local), t_sig)
                j_sig = lower_bound_ge_hit(hit_sl, start, np.int32(n_sl_local), t_sig)

                exit_open = float(exec_open_1m[sig_exit_exec])
                contrib = log_fee_two_sides
                if exit_open > 0.0:
                    if dirn == 1:
                        contrib += float(log_exec_open_1m[sig_exit_exec]) - entry_log
                    else:
                        ratio = exit_open / entry_open
                        if ratio >= 2.0:
                            contrib = NEG_LARGE
                        else:
                            contrib += math.log(2.0 - ratio)

                add_rect(rect_diff, i_sig, j_sig, np.int32(n_tp_local), np.int32(n_sl_local), contrib)

                j_ptr = np.int32(0)
                for i in range(i_sig):
                    t_tp = np.int32(hit_tp[i, start])
                    while j_ptr < j_sig and np.int32(hit_sl[j_ptr, start]) <= t_tp:
                        j_ptr = np.int32(j_ptr + 1)
                    add_row_range(row_diff, np.int32(i), j_ptr, np.int32(n_sl_local), float(log_tp_arr[i]))

                i_ptr = np.int32(0)
                for j in range(j_sig):
                    t_sl = np.int32(hit_sl[j, start])
                    while i_ptr < i_sig and np.int32(hit_tp[i_ptr, start]) < t_sl:
                        i_ptr = np.int32(i_ptr + 1)
                    add_col_range(col_diff, i_ptr, np.int32(n_tp_local), np.int32(j), float(log_sl_arr[j]))
            else:
                i_never = first_equal_hit(hit_tp, start, np.int32(n_tp_local), T_exec)
                j_never = first_equal_hit(hit_sl, start, np.int32(n_sl_local), T_exec)

                if close_on_end == 1 and T_exec > 0:
                    contrib = log_fee_two_sides
                    if last_close_1m > 0.0:
                        if dirn == 1:
                            contrib += log_last_close_1m - entry_log
                        else:
                            ratio = last_close_1m / entry_open
                            if ratio >= 2.0:
                                contrib = NEG_LARGE
                            else:
                                contrib += math.log(2.0 - ratio)
                    add_rect(rect_diff, i_never, j_never, np.int32(n_tp_local), np.int32(n_sl_local), contrib)

                j_ptr = np.int32(0)
                for i in range(i_never):
                    t_tp = np.int32(hit_tp[i, start])
                    while j_ptr < n_sl_local and np.int32(hit_sl[j_ptr, start]) <= t_tp:
                        j_ptr = np.int32(j_ptr + 1)
                    add_row_range(row_diff, np.int32(i), j_ptr, np.int32(n_sl_local), float(log_tp_arr[i]))

                i_ptr = np.int32(0)
                for j in range(j_never):
                    t_sl = np.int32(hit_sl[j, start])
                    while i_ptr < i_never and np.int32(hit_tp[i_ptr, start]) < t_sl:
                        i_ptr = np.int32(i_ptr + 1)
                    add_col_range(col_diff, i_ptr, np.int32(n_tp_local), np.int32(j), float(log_sl_arr[j]))

        for i in range(n_tp_local):
            run = 0.0
            for j in range(n_sl_local):
                run += row_diff[i, j]
                row_diff[i, j] = run

        for j in range(n_sl_local):
            run = 0.0
            for i in range(n_tp_local):
                run += col_diff[i, j]
                col_diff[i, j] = run

        for i in range(n_tp_local):
            row_run = 0.0
            for j in range(n_sl_local):
                row_run += rect_diff[i, j]
                if i == 0:
                    rect_diff[i, j] = row_run
                else:
                    rect_diff[i, j] = row_run + rect_diff[i - 1, j]

        best_log = -1.0e300
        best_tp = np.int32(0)
        best_sl = np.int32(0)
        for tp_i in range(n_tp_local):
            for sl_i in range(n_sl_local):
                value = row_diff[tp_i, sl_i] + col_diff[tp_i, sl_i] + rect_diff[tp_i, sl_i]
                if value > best_log:
                    best_log = value
                    best_tp = np.int32(tp_i)
                    best_sl = np.int32(sl_i)

        out_best_tp_idx[k] = best_tp
        out_best_sl_idx[k] = best_sl

        best_eq = 1.0
        for tr in range(n_trades):
            pf, _, closed = evaluate_trade_factor_hit_tables(
                dir_arr[tr],
                entry_arr[tr],
                sig_exit_arr[tr],
                best_tp,
                best_sl,
                exec_open_1m,
                last_close_1m,
                T_exec,
                close_on_end,
                hit_long_tp,
                hit_long_sl,
                hit_short_tp,
                hit_short_sl,
                long_tp_eq,
                long_sl_eq,
                short_tp_eq,
                short_sl_eq,
            )
            if closed == 1:
                best_eq *= fee_two_sides * pf
        out_best_ret[k] = np.float32(best_eq - 1.0)


In [20]:
@njit_cached(parallel=True, fastmath=True)
def proxy_prefilter_combos_chunk(
    combo_dema_idx: np.ndarray,
    combo_ema_idx: np.ndarray,
    combo_sma_idx: np.ndarray,
    combo_roc_idx: np.ndarray,
    combo_std_idx: np.ndarray,
    dema_eval_T: np.ndarray,
    ema_eval_T: np.ndarray,
    sma_eval_T: np.ndarray,
    roc_eval_T: np.ndarray,
    std_eval_T: np.ndarray,
    ret_15m: np.ndarray,
    min_confirm: np.int32,
    fee_penalty_per_confirm: np.float32,
    out_confirm: np.ndarray,
    out_proxy: np.ndarray,
) -> None:
    """Compute cheap confirmation counts and proxy scores for one combo chunk.

    Parameters:
        combo_*_idx: Row indices into the filtered evaluation matrices.
        *_eval_T: Filtered evaluation matrices shaped `(n_rows, n_signal_intervals)`.
        ret_15m: Close-to-close 15m return vector.
        min_confirm: Minimum number of consensus confirmations required to keep a combo.
        fee_penalty_per_confirm: Linear fee penalty applied to the proxy score.
        out_confirm: Output confirmation-count vector.
        out_proxy: Output proxy-score vector.

    Returns:
        None.

    Assumptions:
        All combo index arrays share the same length.

    Raises:
        None.

    Side effects:
        Writes confirmation counts and proxy scores into the provided arrays.
    """
    K = combo_dema_idx.shape[0]
    n_int = ret_15m.shape[0]

    for k in nb.prange(K):
        di = combo_dema_idx[k]
        ei = combo_ema_idx[k]
        si = combo_sma_idx[k]
        ri = combo_roc_idx[k]
        vi = combo_std_idx[k]

        confirms = np.int32(0)
        proxy = np.float32(0.0)

        for t in range(n_int):
            dirn = consensus_dir5(
                dema_eval_T[di, t],
                ema_eval_T[ei, t],
                sma_eval_T[si, t],
                roc_eval_T[ri, t],
                std_eval_T[vi, t],
            )
            if dirn == 1:
                confirms += 1
                proxy += ret_15m[t]
            elif dirn == -1:
                confirms += 1
                proxy -= ret_15m[t]

        out_confirm[k] = confirms
        if confirms >= min_confirm:
            out_proxy[k] = proxy - fee_penalty_per_confirm * np.float32(confirms)
        else:
            out_proxy[k] = NEG_INF


def iter_combo_chunks(*, local_row_pools, chunk_size: int):
    """Yield bounded combo chunks over filtered local row pools.

    Parameters:
        local_row_pools: Mapping of indicator id to local row indices inside filtered matrices.
        chunk_size: Maximum number of combinations per yielded chunk.

    Returns:
        Generator of dictionaries containing aligned combo index arrays.

    Assumptions:
        Every required indicator id is present in `local_row_pools`.

    Raises:
        None.

    Side effects:
        None.
    """
    required = ("ma.dema", "ma.ema", "ma.sma", "momentum.roc", "volatility.stddev")
    buffers = {indicator_id: [] for indicator_id in required}

    for combo in itertools.product(*(local_row_pools[indicator_id] for indicator_id in required)):
        for indicator_id, value in zip(required, combo):
            buffers[indicator_id].append(int(value))
        if len(buffers[required[0]]) >= chunk_size:
            yield {indicator_id: np.asarray(buffers[indicator_id], dtype=np.int32) for indicator_id in required}
            buffers = {indicator_id: [] for indicator_id in required}

    if buffers[required[0]]:
        yield {indicator_id: np.asarray(buffers[indicator_id], dtype=np.int32) for indicator_id in required}


def run_fast_vs_reference_self_check(*, combo_chunk, indicator_pools, check_n: int = SELF_CHECK_N_DEFAULT, ret_tol: float = 1e-4):
    """Compare the fast monotone TP/SL kernel against the slow reference on a small subset.

    Parameters:
        combo_chunk: Dictionary with combo index arrays over filtered local row pools.
        indicator_pools: Prefiltered indicator pool dictionaries.
        check_n: Maximum number of combinations to compare.
        ret_tol: Allowed absolute return difference between slow and fast outputs.

    Returns:
        Dictionary with the number of checked combos and the maximum absolute return difference.

    Assumptions:
        The chunk already passed combo prefiltering and contains at least one combination.

    Raises:
        AssertionError: If best TP/SL indices differ or the return difference exceeds `ret_tol`.

    Side effects:
        Compiles and executes both TP/SL kernels on the requested subset.
    """
    required = ("ma.dema", "ma.ema", "ma.sma", "momentum.roc", "volatility.stddev")
    n_check = min(check_n, int(combo_chunk[required[0]].shape[0]))
    if n_check <= 0:
        return {"checked": 0, "max_abs_best_ret_diff": 0.0}

    subset = {indicator_id: combo_chunk[indicator_id][:n_check] for indicator_id in required}
    trade_counts = np.empty(n_check, dtype=np.int32)
    count_trades_for_five_combos(
        subset["ma.dema"],
        subset["ma.ema"],
        subset["ma.sma"],
        subset["momentum.roc"],
        subset["volatility.stddev"],
        indicator_pools["ma.dema"]["trade_T"],
        indicator_pools["ma.ema"]["trade_T"],
        indicator_pools["ma.sma"]["trade_T"],
        indicator_pools["momentum.roc"]["trade_T"],
        indicator_pools["volatility.stddev"]["trade_T"],
        sig_entry_exec_idx_15m,
        T_exec_limit_1m,
        trade_counts,
    )

    slow_tp = np.empty(n_check, dtype=np.int32)
    slow_sl = np.empty(n_check, dtype=np.int32)
    slow_ret = np.empty(n_check, dtype=np.float32)
    slow_cnt = np.empty(n_check, dtype=np.int32)

    fast_tp = np.empty(n_check, dtype=np.int32)
    fast_sl = np.empty(n_check, dtype=np.int32)
    fast_ret = np.empty(n_check, dtype=np.float32)
    fast_cnt = np.empty(n_check, dtype=np.int32)

    evaluate_best_tp_sl_trade_list_slow_five(
        subset["ma.dema"],
        subset["ma.ema"],
        subset["ma.sma"],
        subset["momentum.roc"],
        subset["volatility.stddev"],
        indicator_pools["ma.dema"]["trade_T"],
        indicator_pools["ma.ema"]["trade_T"],
        indicator_pools["ma.sma"]["trade_T"],
        indicator_pools["momentum.roc"]["trade_T"],
        indicator_pools["volatility.stddev"]["trade_T"],
        sig_entry_exec_idx_15m,
        price_fields_1m["open"],
        last_close_1m,
        T_exec_limit_1m,
        long_tp_1m,
        long_sl_1m,
        short_tp_1m,
        short_sl_1m,
        long_tp_eq,
        long_sl_eq,
        short_tp_eq,
        short_sl_eq,
        fee_two_sides,
        CLOSE_ON_END,
        slow_tp,
        slow_sl,
        slow_ret,
        slow_cnt,
    )

    evaluate_best_tp_sl_trade_list_fast_monotone_five(
        subset["ma.dema"],
        subset["ma.ema"],
        subset["ma.sma"],
        subset["momentum.roc"],
        subset["volatility.stddev"],
        indicator_pools["ma.dema"]["trade_T"],
        indicator_pools["ma.ema"]["trade_T"],
        indicator_pools["ma.sma"]["trade_T"],
        indicator_pools["momentum.roc"]["trade_T"],
        indicator_pools["volatility.stddev"]["trade_T"],
        sig_entry_exec_idx_15m,
        price_fields_1m["open"],
        last_close_1m,
        T_exec_limit_1m,
        long_tp_1m,
        long_sl_1m,
        short_tp_1m,
        short_sl_1m,
        long_tp_eq,
        long_sl_eq,
        short_tp_eq,
        short_sl_eq,
        trade_counts,
        log_fac_tp_long,
        log_fac_sl_long,
        log_fac_tp_short,
        log_fac_sl_short,
        log_fee_two_sides,
        log_exec_open_1m,
        log_last_close_1m,
        fee_two_sides,
        CLOSE_ON_END,
        fast_tp,
        fast_sl,
        fast_ret,
        fast_cnt,
    )

    if not np.array_equal(slow_tp, fast_tp):
        raise AssertionError("Fast TP indices differ from the slow reference.")
    if not np.array_equal(slow_sl, fast_sl):
        raise AssertionError("Fast SL indices differ from the slow reference.")
    if not np.array_equal(slow_cnt, fast_cnt):
        raise AssertionError("Fast trade counts differ from the slow reference.")

    max_abs_best_ret_diff = float(np.max(np.abs(slow_ret.astype(np.float64) - fast_ret.astype(np.float64))))
    if max_abs_best_ret_diff > ret_tol:
        raise AssertionError(
            f"Fast best return differs from the slow reference by {max_abs_best_ret_diff}, tolerance {ret_tol}."
        )

    return {"checked": n_check, "max_abs_best_ret_diff": max_abs_best_ret_diff}


def search_topk_five_indicator_trade_list(*, row_pools, indicator_top_frac: float = PREFILTER_TOP_FRAC, min_nonzero: int = PREFILTER_MIN_NONZERO, combo_top_frac: float = COMBO_PREFILTER_TOP_FRAC, combo_min_confirm: int = COMBO_MIN_CONFIRM, combo_chunk_size: int = COMBO_CHUNK_SIZE, top_k: int = TOP_K_DEFAULT, self_check_n: int = SELF_CHECK_N_DEFAULT, verbose: bool = True):
    """Run a universal five-indicator search with prefilter, trade lists, and exact TP/SL evaluation.

    Parameters:
        row_pools: Mapping of indicator id to original artifact row ids.
        indicator_top_frac: Fraction of rows to keep in the single-indicator prefilter.
        min_nonzero: Minimum number of non-zero signals required for an indicator row.
        combo_top_frac: Fraction of valid combinations to keep inside each combo chunk before exact evaluation.
        combo_min_confirm: Minimum number of consensus confirmations required before exact evaluation.
        combo_chunk_size: Maximum number of combos per chunk.
        top_k: Number of best exact results to retain globally.
        self_check_n: Number of exact combos to validate against the slow reference.
        verbose: Whether to print diagnostics during the search.

    Returns:
        Dictionary containing filtered pool diagnostics, optional self-check diagnostics, and the global top-k exact results.

    Assumptions:
        `row_pools` contains bounded row selections. The full five-indicator cartesian product remains too large for exhaustive exact search.

    Raises:
        KeyError: If any required indicator pool is missing.
        AssertionError: If the slow-vs-fast self-check fails.

    Side effects:
        Triggers Numba compilation on first use and runs the exact kernels over the selected combo chunks.
    """
    required = ("ma.dema", "ma.ema", "ma.sma", "momentum.roc", "volatility.stddev")
    indicator_pools = {}
    for indicator_id in required:
        requested_rows = np.asarray(row_pools[indicator_id], dtype=np.int32)
        indicator_pools[indicator_id] = prepare_indicator_pool(
            indicator_id=indicator_id,
            row_ids=requested_rows,
            top_frac=indicator_top_frac,
            min_nonzero=min_nonzero,
            fee_rate=FEE_RATE,
            time_chunk=TIME_CHUNK,
        )

    local_row_pools = {
        indicator_id: np.arange(indicator_pools[indicator_id]["trade_T"].shape[0], dtype=np.int32)
        for indicator_id in required
    }

    heap = []
    self_check = None
    total_combo_chunks = 0
    total_exact_candidates = 0
    fee_penalty_per_confirm = np.float32(1.5 * FEE_RATE)

    for combo_chunk in iter_combo_chunks(local_row_pools=local_row_pools, chunk_size=combo_chunk_size):
        total_combo_chunks += 1
        chunk_len = int(combo_chunk[required[0]].shape[0])
        out_confirm = np.empty(chunk_len, dtype=np.int32)
        out_proxy = np.empty(chunk_len, dtype=np.float32)

        proxy_prefilter_combos_chunk(
            combo_chunk["ma.dema"],
            combo_chunk["ma.ema"],
            combo_chunk["ma.sma"],
            combo_chunk["momentum.roc"],
            combo_chunk["volatility.stddev"],
            indicator_pools["ma.dema"]["eval_T"],
            indicator_pools["ma.ema"]["eval_T"],
            indicator_pools["ma.sma"]["eval_T"],
            indicator_pools["momentum.roc"]["eval_T"],
            indicator_pools["volatility.stddev"]["eval_T"],
            signal_returns_15m,
            np.int32(combo_min_confirm),
            fee_penalty_per_confirm,
            out_confirm,
            out_proxy,
        )

        valid_idx = np.flatnonzero(out_proxy > NEG_INF / 2)
        if valid_idx.size == 0:
            continue

        keep_local = topk_fraction_idx(out_proxy[valid_idx], combo_top_frac)
        keep_idx = np.sort(valid_idx[keep_local].astype(np.int32))
        selected = {indicator_id: combo_chunk[indicator_id][keep_idx] for indicator_id in required}
        selected_confirm = out_confirm[keep_idx]
        selected_proxy = out_proxy[keep_idx]
        total_exact_candidates += int(keep_idx.size)

        trade_counts = np.empty(int(keep_idx.size), dtype=np.int32)
        count_trades_for_five_combos(
            selected["ma.dema"],
            selected["ma.ema"],
            selected["ma.sma"],
            selected["momentum.roc"],
            selected["volatility.stddev"],
            indicator_pools["ma.dema"]["trade_T"],
            indicator_pools["ma.ema"]["trade_T"],
            indicator_pools["ma.sma"]["trade_T"],
            indicator_pools["momentum.roc"]["trade_T"],
            indicator_pools["volatility.stddev"]["trade_T"],
            sig_entry_exec_idx_15m,
            T_exec_limit_1m,
            trade_counts,
        )

        if self_check is None and self_check_n > 0:
            self_check = run_fast_vs_reference_self_check(
                combo_chunk=selected,
                indicator_pools=indicator_pools,
                check_n=self_check_n,
            )

        best_tp_idx = np.empty(int(keep_idx.size), dtype=np.int32)
        best_sl_idx = np.empty(int(keep_idx.size), dtype=np.int32)
        best_ret = np.empty(int(keep_idx.size), dtype=np.float32)
        out_trade_counts = np.empty(int(keep_idx.size), dtype=np.int32)

        evaluate_best_tp_sl_trade_list_fast_monotone_five(
            selected["ma.dema"],
            selected["ma.ema"],
            selected["ma.sma"],
            selected["momentum.roc"],
            selected["volatility.stddev"],
            indicator_pools["ma.dema"]["trade_T"],
            indicator_pools["ma.ema"]["trade_T"],
            indicator_pools["ma.sma"]["trade_T"],
            indicator_pools["momentum.roc"]["trade_T"],
            indicator_pools["volatility.stddev"]["trade_T"],
            sig_entry_exec_idx_15m,
            price_fields_1m["open"],
            last_close_1m,
            T_exec_limit_1m,
            long_tp_1m,
            long_sl_1m,
            short_tp_1m,
            short_sl_1m,
            long_tp_eq,
            long_sl_eq,
            short_tp_eq,
            short_sl_eq,
            trade_counts,
            log_fac_tp_long,
            log_fac_sl_long,
            log_fac_tp_short,
            log_fac_sl_short,
            log_fee_two_sides,
            log_exec_open_1m,
            log_last_close_1m,
            fee_two_sides,
            CLOSE_ON_END,
            best_tp_idx,
            best_sl_idx,
            best_ret,
            out_trade_counts,
        )

        for local_idx in range(int(keep_idx.size)):
            score = float(best_ret[local_idx])
            dema_orig_row = int(indicator_pools["ma.dema"]["filtered_row_ids"][selected["ma.dema"][local_idx]])
            ema_orig_row = int(indicator_pools["ma.ema"]["filtered_row_ids"][selected["ma.ema"][local_idx]])
            sma_orig_row = int(indicator_pools["ma.sma"]["filtered_row_ids"][selected["ma.sma"][local_idx]])
            roc_orig_row = int(indicator_pools["momentum.roc"]["filtered_row_ids"][selected["momentum.roc"][local_idx]])
            std_orig_row = int(indicator_pools["volatility.stddev"]["filtered_row_ids"][selected["volatility.stddev"][local_idx]])

            item = {
                "total_return_pct": score * 100.0,
                "confirm_count": int(selected_confirm[local_idx]),
                "proxy_score": float(selected_proxy[local_idx]),
                "trade_count": int(out_trade_counts[local_idx]),
                "best_tp_pct": float(artifact_tp_grid[best_tp_idx[local_idx]] * 100.0),
                "best_sl_pct": float(artifact_sl_grid[best_sl_idx[local_idx]] * 100.0),
                "ma.dema": five_indicator_row_catalogs_15m["ma.dema"][dema_orig_row],
                "ma.ema": five_indicator_row_catalogs_15m["ma.ema"][ema_orig_row],
                "ma.sma": five_indicator_row_catalogs_15m["ma.sma"][sma_orig_row],
                "momentum.roc": five_indicator_row_catalogs_15m["momentum.roc"][roc_orig_row],
                "volatility.stddev": five_indicator_row_catalogs_15m["volatility.stddev"][std_orig_row],
            }
            heap_item = (score, item)
            if len(heap) < top_k:
                heapq.heappush(heap, heap_item)
            elif score > heap[0][0]:
                heapq.heapreplace(heap, heap_item)

    filtered_pool_sizes = {
        indicator_id: int(indicator_pools[indicator_id]["trade_T"].shape[0])
        for indicator_id in required
    }
    if verbose:
        print("filtered pool sizes:", filtered_pool_sizes)
        print("combo chunks processed:", total_combo_chunks)
        print("exact candidates evaluated:", total_exact_candidates)
        if self_check is not None:
            print("self-check:", self_check)

    top_results = [item for _, item in sorted(heap, key=lambda pair: pair[0], reverse=True)]
    return {
        "filtered_pool_sizes": filtered_pool_sizes,
        "combo_chunks_processed": total_combo_chunks,
        "exact_candidates_evaluated": total_exact_candidates,
        "self_check": self_check,
        "top_results": top_results,
    }


close_only_full_rows_5 = {
    indicator_id: row_ids_for_sources(indicator_id=indicator_id, source_names=["close"])
    for indicator_id in FULL_INDICATORS_15M
}
close_only_demo_rows_5 = {
    indicator_id: rows[: min(16, rows.shape[0])]
    for indicator_id, rows in close_only_full_rows_5.items()
}

print("close_only_demo_rows_5 sizes:", {key: int(value.shape[0]) for key, value in close_only_demo_rows_5.items()})


close_only_demo_rows_5 sizes: {'ma.dema': 16, 'ma.ema': 16, 'ma.sma': 16, 'momentum.roc': 10, 'volatility.stddev': 8}


In [21]:
row_pools = {
    "ma.dema": row_ids_for_sources(indicator_id="ma.dema", source_names=["close", "low"])[:300],
    "ma.ema": row_ids_for_sources(indicator_id="ma.ema", source_names=["close", "low"])[:300],
    "ma.sma": row_ids_for_sources(indicator_id="ma.sma", source_names=["close","low"])[:300],
    "momentum.roc": row_ids_for_sources(indicator_id="momentum.roc", source_names=["close", "low"])[:30],
    "volatility.stddev": row_ids_for_sources(indicator_id="volatility.stddev", source_names=["close","low"])[:300],
}

result = search_topk_five_indicator_trade_list(
    row_pools=row_pools,
    min_nonzero=1,
    combo_min_confirm=1,
    top_k=30,
    self_check_n=2,
    verbose=False,
)

In [22]:
result["top_results"][1]

{'total_return_pct': 79.32896018028259,
 'confirm_count': 64476,
 'proxy_score': -35.561309814453125,
 'trade_count': 2471,
 'best_tp_pct': 2.0,
 'best_sl_pct': 2.0,
 'ma.dema': {'indicator_id': 'ma.dema',
  'row_id': 187,
  'source': 'close',
  'window': 192},
 'ma.ema': {'indicator_id': 'ma.ema',
  'row_id': 187,
  'source': 'close',
  'window': 192},
 'ma.sma': {'indicator_id': 'ma.sma',
  'row_id': 653,
  'source': 'low',
  'window': 70},
 'momentum.roc': {'indicator_id': 'momentum.roc',
  'row_id': 37,
  'source': 'low',
  'window': 63},
 'volatility.stddev': {'indicator_id': 'volatility.stddev',
  'row_id': 24,
  'source': 'low',
  'window': 10}}